In [1]:
# Same query, two retrievers. Scores on purposely mismatched scales.
dense = {   # cosine similarity, 0–1
    "D1": 0.91, "D2": 0.88, "D3": 0.85, "D4": 0.71, "D5": 0.66,
}
bm25 = {    # BM25, unbounded ~0–40
    "D3": 38.0, "D6": 31.0, "D1": 12.0, "D7": 9.0, "D2": 4.0,
}

In [3]:
docs = set(dense) | set(bm25)
naive = {d: dense.get(d, 0) + bm25.get(d, 0) for d in docs}
print(sorted(naive.items(), key=lambda x: -x[1]))

[('D3', 38.85), ('D6', 31.0), ('D1', 12.91), ('D7', 9.0), ('D2', 4.88), ('D4', 0.71), ('D5', 0.66)]


In [4]:
def minmax(scores):
    lo, hi = min(scores.values()), max(scores.values())
    return {d: (s - lo) / (hi - lo) for d, s in scores.items()}

nd, nb = minmax(dense), minmax(bm25)
w_dense, w_bm25 = 0.5, 0.5
weighted = {d: w_dense*nd.get(d, 0) + w_bm25*nb.get(d, 0) for d in docs}
print(sorted(weighted.items(), key=lambda x: -x[1]))

[('D3', 0.8799999999999999), ('D1', 0.6176470588235294), ('D2', 0.43999999999999995), ('D6', 0.39705882352941174), ('D4', 0.09999999999999987), ('D7', 0.07352941176470588), ('D5', 0.0)]


In [5]:
def rrf(list_of_rankings, k=60):
    scores = {}
    for ranking in list_of_rankings:              # each ranking = dict, insertion order = rank
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0) + 1 / (k + rank)
    return dict(sorted(scores.items(), key=lambda x: -x[1]))

fused = rrf([dense, bm25], k=60)
print(fused)

{'D1': 0.032266458495966696, 'D3': 0.032266458495966696, 'D2': 0.0315136476426799, 'D6': 0.016129032258064516, 'D4': 0.015625, 'D7': 0.015625, 'D5': 0.015384615384615385}


In [6]:
for k in [1, 5, 10, 60, 100]:
    print(k, rrf([dense, bm25], k=k))

1 {'D1': 0.75, 'D3': 0.75, 'D2': 0.5, 'D6': 0.3333333333333333, 'D4': 0.2, 'D7': 0.2, 'D5': 0.16666666666666666}
5 {'D1': 0.29166666666666663, 'D3': 0.29166666666666663, 'D2': 0.24285714285714285, 'D6': 0.14285714285714285, 'D4': 0.1111111111111111, 'D7': 0.1111111111111111, 'D5': 0.1}
10 {'D1': 0.16783216783216784, 'D3': 0.16783216783216784, 'D2': 0.15, 'D6': 0.08333333333333333, 'D4': 0.07142857142857142, 'D7': 0.07142857142857142, 'D5': 0.06666666666666667}
60 {'D1': 0.032266458495966696, 'D3': 0.032266458495966696, 'D2': 0.0315136476426799, 'D6': 0.016129032258064516, 'D4': 0.015625, 'D7': 0.015625, 'D5': 0.015384615384615385}
100 {'D1': 0.01960972796308757, 'D3': 0.01960972796308757, 'D2': 0.019327731092436976, 'D6': 0.00980392156862745, 'D4': 0.009615384615384616, 'D7': 0.009615384615384616, 'D5': 0.009523809523809525}


RRF: Reciprocal Rank Fusion.. so there are basically two types of retrievals, dense and sparse. Dense retrieval is the retrieval based on smiliarity scores from embeddings like cosine similarity where as sparse retrieval is retrieval based on keywords matching. Each compliment for each other. Dense retrieves chunks that are semantically similar where as the sparse retreival makes sure that the query is directly matching, but the issue is both formats scores their chunks in different scales. Now the requirement is to fuse those results. We take dense scores in cosine similarity (so its around -1 to 1) Cosine similarity is effectively between 0 and 1 in practice. where as sparse may give in other scale like one of famous Sparse technique BM25 gives in ragne 40 - 0. So to fuse these we have majorly 2 types of fusing techniques , Rank based fusing and score based fusing. Score Based fusing is weighted sum. where we normalize the scores to a similar level and do their average sum. but here a missing document gets strict penality of 0. and sometimes docs present in both retirevals gets more score than a relevant chunk present in single retireval , In weighted sum a strong single-retrieval doc gets zero-filled on the missing side and sinks, even if it was highly relevant.. where as Rank based fusion avoid these imbalances by completely eliminating the scores and fusing the chunks based on the ranknig. a doc present in both retrieval gets a better score but that depends on the relative position of doc in 2 retirevals. A doc which is no 1 in one retireval and not present in another retrieval would still make the top 5 in RRF making it a better choice for the Fusing overall.

Mistake 1 : No two words are almost never semantic opposites in vector space so Cosine similarity is effectively between 0 and 1 in practice. because two pieces of real text are almost never semantic opposites in vector space.
Mistake 2 : In weighted sum a strong single-retrieval doc gets zero-filled on the missing side and sinks, even if it was highly relevant.

RRF: Reciprocal Rank Fusion. So there are basically two types of retrievals, dense and sparse. Dense retrieval is based on similarity scores from embeddings like cosine similarity, whereas sparse retrieval is based on keyword matching. Each complements the other — dense retrieves chunks that are semantically similar, whereas sparse makes sure the query directly matches on keywords. But the issue is both score their chunks on different scales. Dense gives cosine similarity, which is effectively 0 to 1 in practice, whereas sparse gives another scale entirely — e.g. BM25, a famous sparse technique, returns unbounded scores roughly in the 0 to 40 range.

Now the requirement is to fuse these two result lists. There are majorly two families of fusion: score-based and rank-based.

Score-based fusion is weighted sum: we normalize both score lists to a comparable range, then take a weighted sum. The problem is normalization is fragile — it's hostage to outliers, and a doc missing from one list gets zero-filled, which is a brutal penalty. So a strong single-retrieval doc gets a 0 on the missing side and sinks, even if it was highly relevant.

Rank-based fusion avoids these imbalances by completely discarding the scores and fusing purely on rank position. Each doc scores 1/(k + rank) in each list, summed across lists — so the scale mismatch simply never matters, because RRF never looks at a score. k is a steepness knob: the standard k=60 (Cormack et al., 2009) softens the gap between top ranks so cross-list agreement counts for more than any single #1. A doc present in both retrievals accumulates two terms and naturally floats up, but that depends on its relative position in each list. And critically, a doc that's #1 in one retrieval and absent from the other can still make the top 5 in RRF — its single strong rank outweighs some other doc's two mediocre ones. That robustness is what makes RRF the better default for fusion.

`k` is a flattening knob in `1/(k+rank)`: small `k` makes the top ranks dominate (being #1 is worth far more than #2), while large `k` flattens the gaps so cross-list agreement matters more than any single top position. The standard `k=60` sits in between — it softens the rank-1 advantage just enough that a doc both retrievers agree on can beat a doc that's #1 in only one list.

`k=60` is an empirical default from Cormack et al. (2009) — they tested different values on retrieval benchmarks and 60 gave the best results, so it stuck as the convention. There's nothing magical about it; it's just a well-tested sweet spot that flattens rank gaps enough to reward cross-list agreement without erasing the advantage of being near the top — and you can tune it if your own eval says otherwise.